In [13]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import numpy as np
import time

In [10]:
# --- Extraction Functions (Same as before) ---
def get_title(soup):
    try:
        title = soup.find("span", attrs={"id":'productTitle'})
        return title.text.strip()
    except AttributeError:
        return ""

def get_price(soup):
    # Amazon uses multiple IDs for prices; checking the most common ones
    for selector in ['priceblock_ourprice', 'priceblock_dealprice', 'price_inside_buybox']:
        try:
            price = soup.find("span", attrs={'id': selector}).string.strip()
            return price
        except:
            continue
    return ""

def get_rating(soup):
    try:
        return soup.find("span", attrs={'class':'a-icon-alt'}).string.strip()
    except:
        return ""

def get_review_count(soup):
    try:
        return soup.find("span", attrs={'id':'acrCustomerReviewText'}).string.strip()
    except:
        return ""

def get_availability(soup):
    try:
        available = soup.find("div", attrs={'id':'availability'}).find("span").string.strip()
        return available
    except:
        return "Not Available"


In [14]:

# --- Main Logic ---
if __name__ == '__main__':
    HEADERS = ({'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36', 'Accept-Language': 'en-US, en;q=0.5'})
    
    links_list = []
    
    # STEP 1: Loop through first 7 pages to gather ~120-140 links
    print("Gathering product links...")
    for page in range(1, 8):
        # We add the &page= parameter to the URL
        URL = f"https://www.amazon.com/s?k=playstation+5&page={page}"
        
        webpage = requests.get(URL, headers=HEADERS)
        soup = BeautifulSoup(webpage.content, "html.parser")
        
        # Find all product links
        links = soup.find_all("a", attrs={'class':'a-link-normal s-no-outline'})
        
        for link in links:
            href = link.get('href')
            if href and href.startswith('/'):
                links_list.append("https://www.amazon.com" + href)
        
        print(f"Page {page} processed. Total links found: {len(links_list)}")
        time.sleep(2) # Pause to avoid being blocked
        
        if len(links_list) >= 120: # Stop gathering links once we have enough
            break

    # STEP 2: Scrape details for each link (Limiting to 110 to be safe)
    d = {"title":[], "price":[], "rating":[], "reviews":[], "availability":[], "link":[]}
    
    print(f"\nStarting deep scrape of {min(len(links_list), 110)} products...")
    
    for i, full_url in enumerate(links_list[:110]):
        try:
            new_webpage = requests.get(full_url, headers=HEADERS)
            new_soup = BeautifulSoup(new_webpage.content, "html.parser")

            d['title'].append(get_title(new_soup))
            d['price'].append(get_price(new_soup))
            d['rating'].append(get_rating(new_soup))
            d['reviews'].append(get_review_count(new_soup))
            d['availability'].append(get_availability(new_soup))
            d['link'].append(full_url)
            
            print(f"Scraped product {i+1}/110")
            time.sleep(1.5) # Crucial delay to avoid CAPTCHA
            
        except Exception as e:
            print(f"Skipping product {i+1} due to error.")
            continue

    # STEP 3: Save to CSV
    amazon_df = pd.DataFrame.from_dict(d)
    amazon_df['title'].replace('', np.nan, inplace=True)
    amazon_df = amazon_df.dropna(subset=['title'])
    
    try:
        amazon_df.to_csv("amazon_data_100.csv", header=True, index=False)
        print(f"\nDone! Saved {len(amazon_df)} valid records to 'amazon_data_100.csv'")
    except PermissionError:
        print("\nError: Close 'amazon_data_100.csv' in Excel and try again.")

Gathering product links...
Page 1 processed. Total links found: 16
Page 2 processed. Total links found: 43
Page 3 processed. Total links found: 59
Page 4 processed. Total links found: 75
Page 5 processed. Total links found: 91
Page 6 processed. Total links found: 107
Page 7 processed. Total links found: 127

Starting deep scrape of 110 products...
Scraped product 1/110
Scraped product 2/110
Scraped product 3/110
Scraped product 4/110
Scraped product 5/110
Scraped product 6/110
Scraped product 7/110
Scraped product 8/110
Scraped product 9/110
Scraped product 10/110
Scraped product 11/110
Scraped product 12/110
Scraped product 13/110
Scraped product 14/110
Scraped product 15/110
Scraped product 16/110
Scraped product 17/110
Scraped product 18/110
Scraped product 19/110
Scraped product 20/110
Scraped product 21/110
Scraped product 22/110
Scraped product 23/110
Scraped product 24/110
Scraped product 25/110
Scraped product 26/110
Scraped product 27/110
Scraped product 28/110
Scraped product

In [9]:
amazon_df

,title,price,rating,reviews,availability,link
0,PS5 - Sony PlayStation 5 Digital Edition Gamin...,,4.5 out of 5 stars,(273),Not Available,https://www.amazon.com/PS5-Playstation-Digital...
1,PlayStation 5 Disc Edition Console (slim),,4.5 out of 5 stars,"(5,249)",Only 11 left in stock - order soon.,https://www.amazon.com/PlayStation%C2%AE5-cons...
2,Sony PlayStation 5 PS5 Disc Version Gaming Con...,,4.5 out of 5 stars,(6),Not Available,https://www.amazon.com/Sony-Playstation-Versio...
3,PlayStation 5 Console – NBA 2K26 Bundle (model...,,4.6 out of 5 stars,(202),Not Available,https://www.amazon.com/PlayStation-Console-Bun...
4,PlayStation PS5 Console – God of War Ragnarök ...,,4.5 out of 5 stars,"(13,894)",Not Available,https://www.amazon.com/PlayStation-PS5-Console...
5,PlayStation 5 Console (PS5),,4.5 out of 5 stars,"(9,172)",Only 1 left in stock - order soon.,https://www.amazon.com/PlayStation-5-Console-C...
6,PlayStation 5 Console – Marvel’s Spider-Man 2 ...,,4.5 out of 5 stars,(278),Only 1 left in stock - order soon.,https://www.amazon.com/PlayStation-Console-Mar...
7,PlayStation DualSense™ Wireless Controller - f...,,4.3 out of 5 stars,"(12,914)",In Stock,https://www.amazon.com/PlayStation-DualSenseTM...
8,Marvel's Spider-Man: Miles Morales - PlayStati...,,4.7 out of 5 stars,"(1,456)",In Stock,https://www.amazon.com/Marvels-Spider-Man-Mile...
9,REANIMAL - PlayStation 5,,4.4 out of 5 stars,(48),In Stock,https://www.amazon.com/REANIMAL-PlayStation-5/...
